In [1]:
import numpy
import re
import math
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
import numpy
import math


import re
import numpy
import math


#Calculate m/z of chemical derivatization when different conditions are used.
#TODO 这个function以后可以弃用了
def chemical_deriv_conditions(chemical_deriv):
    mz_cd = 0
    if "-" in chemical_deriv:
        chemical_deriv_list = str.split(chemical_deriv, "-")
        if len(chemical_deriv_list) == 2:
            mz_cd += init_chemical_deriv(chemical_deriv_list[0])
            mz_cd -= init_chemical_deriv(chemical_deriv_list[1])
        else:
            print("Wrong chemical derivatization formula")
    else:
        mz_cd += init_chemical_deriv(chemical_deriv)

    return mz_cd

'''
count number of specific atom
Input:
formula: (formula) string
atom: (specific atom) string
Output:
sum_atom: (sum of specific atom) int
'''
def count_atom(formula, atom):
    sum_atom = 0
    #报错在formula_deconvolution中写
    for i in re.finditer(atom, formula):
        start_index = i.start() + 1
        atom_match = re.match("\d+", formula[start_index:])
        if atom_match:
            sum_atom += int(atom_match.group())
        else:
            sum_atom += 1

    return sum_atom

'''
calculating the m/z of adduct

Input:
adduct (adduct of lipid PB product) string

Ouput:
mz_adduct: (m/z of adduct of lipid PB product) float
#TODO Here, we suppose the adduct of PB diagnostics are Hydrogen
'''
def init_adduct(adduct):
    mz_H = 1.007825
    mz_N = 14.00307
    mz_C = 12
    mz_O = 15.9949
    mz_adduct = -10
    if adduct == "H+":
        mz_adduct = mz_H
    if adduct == "NH4+":
        mz_adduct = mz_N + mz_H * 4
    if adduct == "H-":
        mz_adduct = -mz_H
    if adduct == "HCO3-":
        mz_adduct = mz_H + mz_C + mz_O * 3
    if adduct == "Na+":
        mz_adduct = 22.9898

    if mz_adduct == -10:
        print("adduct is not in the list")

    return mz_adduct

'''
deconvolute formula of whatever
Input:
formula: (formula) string
Output:
formula_dec: (deconvoluted formula) list[] dict{"element": string, "num": int}
'''
def formula_deconvolution(formula):
    formula_dec = []
    atom_list = ["C", "H", "O", "N", "P", "F"]
    for atom in atom_list:
        if re.search(atom, formula):
            dict_element = {"element": atom, "num": count_atom(formula, atom)}
            formula_dec.append(dict_element)

    return formula_dec

'''
Calculating S/N ratio in a spectra

Input:
inten_list: (intensity list of a spectra) list[] float

Output:
[float baseline,float std]: (baseline and std of a spectra)
''' 
def SNR_analysis(inten_list):
    Real_inten=inten_list[inten_list > numpy.min(inten_list)] # 仅保留大于列表中最小强度的值。这一步有效地去除了噪声或非常低强度的数据点
    if len(Real_inten):
        index_cycle=20
        Inten_baseline=numpy.mean(Real_inten) # 计算基线
        Inten_std=numpy.std(numpy.array(Real_inten[Real_inten<Inten_baseline])) # 计算低于基线的强度值的标准差，它是噪声水平的估算值
        while index_cycle > 0:
            Mid_inten=Real_inten[Real_inten<(Inten_baseline+3*Inten_std)]
            if len(Mid_inten) == 0:
                break
            if numpy.std(Mid_inten) == Inten_std or numpy.std(Mid_inten) == 0:
                break
            Inten_baseline=numpy.mean(Mid_inten)
            Inten_std=numpy.std(Mid_inten)
            index_cycle -= 1

        return Inten_baseline/numpy.max(Real_inten),Inten_std/numpy.max(Real_inten)
        # 基线，标准差与最大强度值的比值
    else:
        return 0, 0

'''
calculate mass weight of chemical derivatization
Input:
chemical_deriv_name: (name of chemical derivatization name) string
Output:
mz_chemical_deriv: (mass weight of chemical derivatization) float
'''
def init_chemical_deriv(chemical_deriv_name):
    mz_cl = 35.9683
    mz_C = 12
    mz_H = 1.0078
    mz_O = 15.9979
    mz_N = 14.0031
    mz_P = 30.9738
    mz_F = 18.998
    mz_S = 33.9721
    mz_chemical_deriv = 0
    formula_chemical_deriv = []
    if "-" not in chemical_deriv_name:
        chemical_formula_list = formula_deconvolution(chemical_deriv_name)
        formula_chemical_deriv.append(chemical_deriv_name)
        atom_list = ["cl", "C", "H", "O", "N", "P", "S", "F"]
        mz_atom_list = [mz_cl, mz_C, mz_H, mz_O, mz_N, mz_P, mz_S, mz_F]
        for index_atom in range(len(atom_list)):
            for chemical_formula in chemical_formula_list:
                if atom_list[index_atom] == chemical_formula["element"]:
                    mz_chemical_deriv += mz_atom_list[index_atom] * chemical_formula["num"]

    else:
        chemical_formula_list = formula_deconvolution(chemical_deriv_name.split("-")[0])
        formula_chemical_deriv.append(chemical_deriv_name.split("-")[0])
        atom_list = ["cl", "C", "H", "O", "N", "P", "S", "F"]
        mz_atom_list = [mz_cl, mz_C, mz_H, mz_O, mz_N, mz_P, mz_S, mz_F]
        for index_atom in range(len(atom_list)):
            for chemical_formula in chemical_formula_list:
                if atom_list[index_atom] == chemical_formula["element"]:
                    mz_chemical_deriv += mz_atom_list[index_atom] * chemical_formula["num"]
                
        chemical_formula_list = formula_deconvolution(chemical_deriv_name.split("-")[1])
        formula_chemical_deriv.append(chemical_deriv_name.split("-")[1])
        atom_list = ["cl", "C", "H", "O", "N", "P", "S", "F"]
        mz_atom_list = [mz_cl, mz_C, mz_H, mz_O, mz_N, mz_P, mz_S, mz_F]
        for index_atom in range(len(atom_list)):
            for chemical_formula in chemical_formula_list:
                if atom_list[index_atom] == chemical_formula["element"]:
                    mz_chemical_deriv -= mz_atom_list[index_atom] * chemical_formula["num"]

    return mz_chemical_deriv, formula_chemical_deriv

'''
list peaks according to their precursors
Input:
data_peak: (data) list[] {"precursor": float, "mz": list[] float, "inten": list[] float}
Output:
redata_peak:  (data) list[] {"precursor": float, "mz": list[] float, "inten": list[] float}
'''
def list_precursor(data_peak):
    redata_peak = []
    peak_list = []
    for data_single in data_peak:
        peak_list.append(data_single["precursor"])

    peak_list.sort()
    for peak_single in peak_list:
        for data_single in data_peak:
            if peak_single == data_single["precursor"]:
                redata_peak.append(data_single)
                break

    return redata_peak

def Dot_product(measure_list,standard_list):
    result = 0
    measure_list = numpy.array(measure_list)
    standard_list = numpy.array(standard_list)
    if sum(measure_list)!=0:
        result=numpy.sum(measure_list*standard_list)/math.sqrt(numpy.sum(measure_list ** 2))/math.sqrt(numpy.sum(standard_list ** 2))

    return round(result,2)


'''
initialize isotope pattern of elements
#TODO the maximum mass difference between +1Da and monoisotope peak is 1.006 in "H" and minimum is 0.997 in "N". Also considering that mass difference between the +2 Da and monoisotope peak is 2.001 in "O", we deomostrate that there are two +2Da peaks having mass difference higher than 0.01 than m/z of lipid with one more degree of unsaturation. These peaks have a "O18" or two "H2".
Onput:
element_pattern: (pattern of C, H, O, N, P) list[] dict{"element": string, "mz_mono": float, "concen_mono": float, "mz_1plus": float, "concen_1plus": float, "mz_2plus": float, "concen_2plus": float}
'''
def init_isotope():
    element_pattern = []
    dict_c = {"element": "C", "mz_mono": 12, "concen_mono": 98.9/100, "mz_1plus": 13.0033, "concen_1plus": 1.1/100, "mz_2plus": None, "concen_2plus": None}
    dict_h = {"element": "H", "mz_mono": 1.0078, "concen_mono": 99.985/100, "mz_1plus": 2.0141, "concen_1plus": 0.015/100, "mz_2plus": None, "concen_2plus": None}
    dict_o = {"element": "O", "mz_mono": 15.9949, "concen_mono": 99.76/100, "mz_1plus": 16.9991, "concen_1plus": 0.038/100, "mz_2plus": 17.9992, "concen_2plus": 0.2/100}
    dict_n = {"element": "N", "mz_mono": 14.0031, "concen_mono": 99.63/100, "mz_1plus": 15.0001, "concen_1plus": 0.37/100, "mz_2plus": None, "concen_2plus": None}
    dict_p = {"element": "P", "mz_mono": 30.9738, "concen_mono": 100/100, "mz_1plus": None, "concen_1plus": None, "mz_2plus": None, "concen_2plus": None}
    element_pattern.append(dict_c)
    element_pattern.append(dict_h)
    element_pattern.append(dict_o)
    element_pattern.append(dict_n)
    element_pattern.append(dict_p)
    return element_pattern

'''
calculate isotope pattern of monoisotope, +1Da and +2Da peak
#TODO currently only consider +2Da isotope
#TODO no error warning here
Input:
lipid formula: (formula of lipid such as C40H84O8NP)
Output:
inten_pattern: (intensity pattern) 1D numpy array
'''
def isotope_pattern(lipid_formula):
    init_iso = init_isotope()
    inten_pattern = numpy.ones(3)
    formula_dec = formula_deconvolution(lipid_formula)
    atom_list = []
    for atom in formula_dec:
        for init_atom in init_iso:
            if atom["element"] == init_atom["element"]:
                atom["concen_mono"] = 0
                atom["concen_1plus"] = 0
                atom["concen_2plus"] = 0
                atom["concen_mono"] = math.pow(init_atom["concen_mono"], atom["num"])
                #calculating +1Da isotope peak for each atom
                if init_atom["mz_1plus"]:
                    atom["concen_1plus"] = atom["num"] * math.pow(init_atom["concen_mono"], atom["num"] - 1)  * init_atom["concen_1plus"]

                #calculating +2Da isotope peak for each atom
                if init_atom["mz_1plus"] and init_atom["mz_2plus"]:
                    atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2) + math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]
                if init_atom["mz_1plus"] and not init_atom["mz_2plus"]:
                    atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2)
                if init_atom["mz_2plus"] and not init_atom["mz_1plus"]:
                     atom["concen_2plus"] = math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]

                atom_list.append(atom)
                break

    for atom in atom_list:
        inten_pattern[0] *= atom["concen_mono"]

    inten_pattern[1] = 0
    for atom in atom_list:
        inten_pattern[1] += inten_pattern[0] * atom["concen_1plus"] / atom["concen_mono"]

    inten_pattern[2] = 0
    for atom in atom_list:
        inten_pattern[2] += inten_pattern[0] * atom["concen_2plus"] / atom["concen_mono"]

    for index_1 in range(len(atom_list)):
        for index_2 in range(len(atom_list)):
            if index_1 != index_2:
                atom1 = atom_list[index_1]
                atom2 = atom_list[index_2]
                inten_pattern[2] += inten_pattern[0] * atom1["concen_1plus"] / atom1["concen_mono"]* atom2["concen_1plus"] / atom2["concen_mono"] / 2#此处/2是因为index_1和index_2的遍历导致的

    return inten_pattern

'''
provide array to draw isotope pattern
Input:
mz: (m/z of monoisotope peak) float
inten: (inten of monoisotpoe peak) float
isotope pattern: (isotope pattern of lipid) 1D numpy array
Output:
isotope_result: (result of isotope array) 2D numpy array
'''
def isotope_array(mz, inten, isotope_pattern):
    mz_H = 2.0141 - 1.0078
    mz_pattern = numpy.zeros(9)
    mz_pattern[1] = mz
    mz_pattern[0] = mz_pattern[1] - 0.01
    mz_pattern[2] = mz_pattern[1] + 0.01
    mz_pattern[3: 6] = mz_pattern[0: 3] + mz_H
    mz_pattern[6: 9] = mz_pattern[3: 6] + mz_H
    inten_rpattern = numpy.zeros(9)
    inten_rpattern[1] = inten
    inten_rpattern[4] = inten / isotope_pattern[0] * isotope_pattern[1]
    inten_rpattern[7] = inten / isotope_pattern[0] * isotope_pattern[2]
    isotope_result = numpy.zeros([9, 2])
    isotope_result[:, 0] = mz_pattern
    isotope_result[:, 1] = inten_rpattern
    return isotope_result

'\ncount number of specific atom\nInput:\nformula: (formula) string\natom: (specific atom) string\nOutput:\nsum_atom: (sum of specific atom) int\n'

'\ncalculating the m/z of adduct\n\nInput:\nadduct (adduct of lipid PB product) string\n\nOuput:\nmz_adduct: (m/z of adduct of lipid PB product) float\n#TODO Here, we suppose the adduct of PB diagnostics are Hydrogen\n'

'\ndeconvolute formula of whatever\nInput:\nformula: (formula) string\nOutput:\nformula_dec: (deconvoluted formula) list[] dict{"element": string, "num": int}\n'

'\nCalculating S/N ratio in a spectra\n\nInput:\ninten_list: (intensity list of a spectra) list[] float\n\nOutput:\n[float baseline,float std]: (baseline and std of a spectra)\n'

'\ncalculate mass weight of chemical derivatization\nInput:\nchemical_deriv_name: (name of chemical derivatization name) string\nOutput:\nmz_chemical_deriv: (mass weight of chemical derivatization) float\n'

'\nlist peaks according to their precursors\nInput:\ndata_peak: (data) list[] {"precursor": float, "mz": list[] float, "inten": list[] float}\nOutput:\nredata_peak:  (data) list[] {"precursor": float, "mz": list[] float, "inten": list[] float}\n'

'\ninitialize isotope pattern of elements\n#TODO the maximum mass difference between +1Da and monoisotope peak is 1.006 in "H" and minimum is 0.997 in "N". Also considering that mass difference between the +2 Da and monoisotope peak is 2.001 in "O", we deomostrate that there are two +2Da peaks having mass difference higher than 0.01 than m/z of lipid with one more degree of unsaturation. These peaks have a "O18" or two "H2".\nOnput:\nelement_pattern: (pattern of C, H, O, N, P) list[] dict{"element": string, "mz_mono": float, "concen_mono": float, "mz_1plus": float, "concen_1plus": float, "mz_2plus": float, "concen_2plus": float}\n'

'\ncalculate isotope pattern of monoisotope, +1Da and +2Da peak\n#TODO currently only consider +2Da isotope\n#TODO no error warning here\nInput:\nlipid formula: (formula of lipid such as C40H84O8NP)\nOutput:\ninten_pattern: (intensity pattern) 1D numpy array\n'

'\nprovide array to draw isotope pattern\nInput:\nmz: (m/z of monoisotope peak) float\ninten: (inten of monoisotpoe peak) float\nisotope pattern: (isotope pattern of lipid) 1D numpy array\nOutput:\nisotope_result: (result of isotope array) 2D numpy array\n'

In [4]:
lipid_formula='C40H84O8NP'
isotope_pattern(lipid_formula)

array([0.62004156, 0.28785849, 0.07522537])

In [7]:
def init_isotope():
    element_pattern = []
    dict_c = {"element": "C", "mz_mono": 12, "concen_mono": 98.9/100, "mz_1plus": 13.0033, "concen_1plus": 1.1/100, "mz_2plus": None, "concen_2plus": None}
    dict_h = {"element": "H", "mz_mono": 1.0078, "concen_mono": 99.985/100, "mz_1plus": 2.0141, "concen_1plus": 0.015/100, "mz_2plus": None, "concen_2plus": None}
    dict_o = {"element": "O", "mz_mono": 15.9949, "concen_mono": 99.76/100, "mz_1plus": 16.9991, "concen_1plus": 0.038/100, "mz_2plus": 17.9992, "concen_2plus": 0.2/100}
    dict_n = {"element": "N", "mz_mono": 14.0031, "concen_mono": 99.63/100, "mz_1plus": 15.0001, "concen_1plus": 0.37/100, "mz_2plus": None, "concen_2plus": None}
    dict_p = {"element": "P", "mz_mono": 30.9738, "concen_mono": 100/100, "mz_1plus": None, "concen_1plus": None, "mz_2plus": None, "concen_2plus": None}
    element_pattern.append(dict_c)
    element_pattern.append(dict_h)
    element_pattern.append(dict_o)
    element_pattern.append(dict_n)
    element_pattern.append(dict_p)
    return element_pattern
init_iso = init_isotope()
init_iso

[{'element': 'C',
  'mz_mono': 12,
  'concen_mono': 0.9890000000000001,
  'mz_1plus': 13.0033,
  'concen_1plus': 0.011000000000000001,
  'mz_2plus': None,
  'concen_2plus': None},
 {'element': 'H',
  'mz_mono': 1.0078,
  'concen_mono': 0.99985,
  'mz_1plus': 2.0141,
  'concen_1plus': 0.00015,
  'mz_2plus': None,
  'concen_2plus': None},
 {'element': 'O',
  'mz_mono': 15.9949,
  'concen_mono': 0.9976,
  'mz_1plus': 16.9991,
  'concen_1plus': 0.00037999999999999997,
  'mz_2plus': 17.9992,
  'concen_2plus': 0.002},
 {'element': 'N',
  'mz_mono': 14.0031,
  'concen_mono': 0.9963,
  'mz_1plus': 15.0001,
  'concen_1plus': 0.0037,
  'mz_2plus': None,
  'concen_2plus': None},
 {'element': 'P',
  'mz_mono': 30.9738,
  'concen_mono': 1.0,
  'mz_1plus': None,
  'concen_1plus': None,
  'mz_2plus': None,
  'concen_2plus': None}]

In [8]:
inten_pattern = numpy.ones(3)
inten_pattern

array([1., 1., 1.])

In [12]:
def formula_deconvolution(formula):
    formula_dec = []
    atom_list = ["C", "H", "O", "N", "P", "F"]
    for atom in atom_list:
        if re.search(atom, formula):
            dict_element = {"element": atom, "num": count_atom(formula, atom)}
            formula_dec.append(dict_element)

    return formula_dec
lipid_formula='C40H84O8NP'
formula_dec = formula_deconvolution(lipid_formula)
formula_dec
type(formula_dec)
atom_list = []

[{'element': 'C', 'num': 40},
 {'element': 'H', 'num': 84},
 {'element': 'O', 'num': 8},
 {'element': 'N', 'num': 1},
 {'element': 'P', 'num': 1}]

list

In [15]:
atom=formula_dec[0]
init_atom=init_iso[0]

In [16]:
atom["concen_mono"] = 0
atom["concen_1plus"] = 0
atom["concen_2plus"] = 0
atom["concen_mono"] = math.pow(init_atom["concen_mono"], atom["num"])
if init_atom["mz_1plus"]:
    atom["concen_1plus"] = atom["num"] * math.pow(init_atom["concen_mono"], atom["num"] - 1)  * init_atom["concen_1plus"]

if init_atom["mz_1plus"] and init_atom["mz_2plus"]:
    atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2) + math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]
if init_atom["mz_1plus"] and not init_atom["mz_2plus"]:
    atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2)
if init_atom["mz_2plus"] and not init_atom["mz_1plus"]:
     atom["concen_2plus"] = math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]

atom_list.append(atom)
atom_list

[{'element': 'C',
  'num': 40,
  'concen_mono': 0.6424682406194554,
  'concen_1plus': 0.2858301576062289,
  'concen_2plus': 0.061992486154232654}]

In [19]:
atom_list=[]
for atom in formula_dec:
    for init_atom in init_iso:
        if atom["element"] == init_atom["element"]:
            atom["concen_mono"] = 0
            atom["concen_1plus"] = 0
            atom["concen_2plus"] = 0
            atom["concen_mono"] = math.pow(init_atom["concen_mono"], atom["num"])
            #calculating +1Da isotope peak for each atom
            if init_atom["mz_1plus"]:
                atom["concen_1plus"] = atom["num"] * math.pow(init_atom["concen_mono"], atom["num"] - 1)  * init_atom["concen_1plus"]

            #calculating +2Da isotope peak for each atom
            if init_atom["mz_1plus"] and init_atom["mz_2plus"]:
                atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2) + math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]
            if init_atom["mz_1plus"] and not init_atom["mz_2plus"]:
                atom["concen_2plus"] = atom["num"] * (atom["num"] - 1) / 2 * math.pow(init_atom["concen_mono"], atom["num"] - 2) * math.pow(init_atom["concen_1plus"], 2)
            if init_atom["mz_2plus"] and not init_atom["mz_1plus"]:
                 atom["concen_2plus"] = math.pow(init_atom["concen_mono"], atom["num"] - 1) * atom["num"] * init_atom["concen_2plus"]

            atom_list.append(atom)
            break
atom_list

[{'element': 'C',
  'num': 40,
  'concen_mono': 0.6424682406194554,
  'concen_1plus': 0.2858301576062289,
  'concen_2plus': 0.061992486154232654},
 {'element': 'H',
  'num': 84,
  'concen_mono': 0.9874781143909716,
  'concen_1plus': 0.012444090854954482,
  'concen_2plus': 7.747608698513942e-05},
 {'element': 'O',
  'num': 8,
  'concen_mono': 0.9809605081739786,
  'concen_1plus': 0.0029892942510514183,
  'concen_2plus': 0.015737112963249356},
 {'element': 'N',
  'num': 1,
  'concen_mono': 0.9963,
  'concen_1plus': 0.0037,
  'concen_2plus': 0.0},
 {'element': 'P',
  'num': 1,
  'concen_mono': 1.0,
  'concen_1plus': 0,
  'concen_2plus': 0}]

In [20]:
for atom in atom_list:
    inten_pattern[0] *= atom["concen_mono"]
inten_pattern

array([0.62004156, 1.        , 1.        ])

In [ ]:
inten_pattern[1] = 0
for atom in atom_list:
    inten_pattern[1] += inten_pattern[0] * atom["concen_1plus"] / atom["concen_mono"]